# KitchenFlow-v2: RL Training Notebook

**Ghost Kitchen Dispatcher — OpenEnv Hackathon Finals**

This notebook:
1. Installs dependencies
2. Runs the naive baseline to get the 'before' score
3. Trains a Qwen-1.5B model with GRPO on L1 → L2 curriculum
4. Plots reward curves (before vs after)
5. Shows before/after qualitative comparison

**Runtime:** T4 GPU recommended. Estimated time: ~30 min for 100 episodes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/kitchenflow-v2/blob/main/KitchenFlow_v2_Training.ipynb)

In [ ]:
# Install dependencies
!pip install unsloth trl torch datasets transformers accelerate bitsandbytes matplotlib -q
!pip install openenv -q  # OpenEnv framework
print('Done.')

In [ ]:
# Clone or upload the env files
# If running from HuggingFace Space, files are already present.
# Otherwise:
# !git clone https://huggingface.co/spaces/YOUR_USERNAME/kitchenflow-v2
# %cd kitchenflow-v2
import os
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ─── 1. Baseline benchmark ───────────────────────────────────────────────
from env import KitchenFlowEnv
from baseline import BaselineDispatcher, TrafficAwareBaseline

def benchmark_agent(agent_cls, difficulty=1, chaos=0, orders=1, n=20):
    scores = []
    for ep in range(n):
        env = KitchenFlowEnv(difficulty=difficulty, chaos_level=chaos,
                              num_orders=orders, seed=1000+ep)
        agent = agent_cls()
        obs = env.reset()
        agent.reset()
        total, done = 0.0, False
        while not done:
            actions = agent.act(obs)
            obs, r, done, _ = env.step(actions)
            total += r
        scores.append(total)
    return scores

print('Running baselines...')
naive_scores = benchmark_agent(BaselineDispatcher, n=20)
smart_scores = benchmark_agent(TrafficAwareBaseline, n=20)

naive_avg = sum(naive_scores)/len(naive_scores)
smart_avg = sum(smart_scores)/len(smart_scores)
print(f'Naive baseline avg:         {naive_avg:.2f}')
print(f'Traffic-aware baseline avg: {smart_avg:.2f}')
print(f'→ RL target: beat {smart_avg:.2f}')

In [ ]:
# ─── 2. Load model ───────────────────────────────────────────────────────
from unsloth import FastLanguageModel

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'  # T4-safe; use 7B on A100

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
)
FastLanguageModel.for_training(model)
print(f'Model loaded: {MODEL_NAME}')

In [ ]:
# ─── 3. Training loop ────────────────────────────────────────────────────
import json, torch
from collections import deque
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
from train import SYSTEM_PROMPT, obs_to_prompt, parse_llm_response, compute_total_reward, rollout

def model_fn(prompt):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outs = model.generate(**inputs, max_new_tokens=128, temperature=0.8,
                               do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

config = GRPOConfig(
    output_dir                  = './checkpoints/kf_colab',
    num_train_epochs            = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-5,
    logging_steps               = 5,
    report_to                   = 'none',
)

N_EPISODES     = 100
all_rewards    = []
recent_rewards = deque(maxlen=10)
current_diff   = 1
curriculum_gates = {1: 40.0, 2: 35.0}

for ep in range(N_EPISODES):
    env  = KitchenFlowEnv(difficulty=current_diff, chaos_level=max(0, current_diff-1),
                           num_orders=1, seed=ep+42)
    roll = rollout(env, model_fn)

    all_rewards.append(roll['total_reward'])
    recent_rewards.append(roll['total_reward'])

    dataset = Dataset.from_dict({
        'prompt':     [SYSTEM_PROMPT + '\n' + p for p in roll['prompts']],
        'completion': roll['responses'],
        'reward':     roll['rewards'],
    })

    # *** ACTUAL TRAINING STEP ***
    trainer = GRPOTrainer(model=model, tokenizer=tokenizer,
                           train_dataset=dataset, args=config, reward_funcs=[])
    trainer.train()

    if ep % 10 == 0:
        avg = sum(recent_rewards)/len(recent_rewards)
        print(f'ep={ep:03d} | avg10={avg:.2f} | baseline={naive_avg:.2f} | diff={current_diff}')
        if len(recent_rewards)==10 and avg >= curriculum_gates.get(current_diff, 999) and current_diff < 3:
            current_diff += 1
            print(f'  *** Advanced to L{current_diff} ***')

print('Training complete!')

In [ ]:
# ─── 4. Reward curve plot ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(all_rewards, alpha=0.35, color='#aaa', linewidth=0.8, label='episode reward')
if len(all_rewards) >= 10:
    roll_mean = np.convolve(all_rewards, np.ones(10)/10, mode='valid')
    ax.plot(range(9, len(all_rewards)), roll_mean, color='#1a7abf', linewidth=2, label='10-ep rolling mean')
ax.axhline(naive_avg, color='#d84040', linewidth=1.5, linestyle='--', label=f'naive baseline ({naive_avg:.1f})')
ax.axhline(smart_avg, color='#e09020', linewidth=1.5, linestyle='--', label=f'traffic-aware baseline ({smart_avg:.1f})')
ax.set_xlabel('Episode')
ax.set_ylabel('Total episode reward')
ax.set_title('KitchenFlow-v2 — Reward Curve')
ax.legend()
ax.grid(alpha=0.3)

mid = len(all_rewards)//2
ax2 = axes[1]
ax2.hist(all_rewards[:mid], bins=15, alpha=0.6, color='#d84040', label=f'first {mid} eps')
ax2.hist(all_rewards[mid:], bins=15, alpha=0.6, color='#1a7abf', label=f'last {len(all_rewards)-mid} eps')
ax2.axvline(naive_avg, color='#888', linewidth=1.5, linestyle='--', label=f'baseline ({naive_avg:.1f})')
ax2.set_xlabel('Episode reward')
ax2.set_ylabel('Count')
ax2.set_title('Score distribution: first vs last half')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as reward_curve.png')

In [ ]:
# ─── 5. Before/After qualitative comparison ──────────────────────────────
from baseline import BaselineDispatcher

TEST_SEED = 777

# Before: naive baseline
env_b   = KitchenFlowEnv(difficulty=2, chaos_level=1, num_orders=1, seed=TEST_SEED)
agent_b = BaselineDispatcher()
obs_b   = env_b.reset()
agent_b.reset()
total_b, done = 0.0, False
print('=== BEFORE (naive baseline) ===')
while not done:
    actions = agent_b.act(obs_b)
    obs_b, r, done, info = env_b.step(actions)
    total_b += r
    if info.get('events'):
        print(f'  step={env_b._env_state.step:02d} | {actions} | reward={r:+.1f} | {info["events"]}')
print(f'BASELINE SCORE: {total_b:.2f}\n')

# After: trained model
env_a = KitchenFlowEnv(difficulty=2, chaos_level=1, num_orders=1, seed=TEST_SEED)
obs_a = env_a.reset()
total_a, done = 0.0, False
print('=== AFTER (trained model) ===')
while not done:
    active_ids = [o['order_id'] for o in obs_a['orders']
                  if not o.get('delivered') and not o.get('failed')]
    if not active_ids:
        _, _, done, _ = env_a.step({})
        continue
    prompt   = obs_to_prompt(obs_a)
    response = model_fn(prompt)
    actions  = parse_llm_response(response, active_ids)
    obs_a, r, done, info = env_a.step(actions)
    total_a += r
    if info.get('events'):
        print(f'  step={env_a._env_state.step:02d} | {actions} | reward={r:+.1f} | {info["events"]}')
print(f'TRAINED SCORE: {total_a:.2f}')
print(f'Improvement: {total_a - total_b:+.2f} ({((total_a-total_b)/abs(total_b)*100) if total_b!=0 else 0:+.1f}%)')

In [ ]:
# ─── 6. Save model ───────────────────────────────────────────────────────
import os
os.makedirs('./checkpoints/merged', exist_ok=True)

# Correct save method — avoids QLoRA 4-bit upcast corruption
model.save_pretrained_merged('./checkpoints/merged', tokenizer, save_method='merged_16bit')
print('Model saved to ./checkpoints/merged')
print('Upload to HuggingFace Hub with: model.push_to_hub("YOUR_USERNAME/kitchenflow-v2-rl")')